# Validacio dels datasets de modelatge

Aquest notebook no te com a objectiu fer una analisi substantiva de la gentrificacio.
La seva funcio es validar si els datasets finals estan preparats per a la fase de clustering.

Objectius:
- comprovar coherencia estructural entre datasets
- detectar valors nuls, infinits i possibles errors de calcul
- revisar distribucions, asimetries i outliers
- documentar decisions abans del clustering

# Llibreries i configuracio

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results' / 'figs'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Carrega de dades

In [ ]:
dim_barris = pd.read_csv(DATA_DIR / 'dimensions' / 'BarcelonaCiutat_Barris.csv')
df_2015 = pd.read_csv(DATA_DIR / 'modelling' / 'df_2015.csv')
df_2023 = pd.read_csv(DATA_DIR / 'modelling' / 'df_2023.csv')
df_deltes = pd.read_csv(DATA_DIR / 'modelling' / 'df_deltes.csv')

print('dim_barris:', dim_barris.shape)
print('df_2015:', df_2015.shape)
print('df_2023:', df_2023.shape)
print('df_deltes:', df_deltes.shape)

# Funcions auxiliars

In [ ]:
def resum_estructura(df, nom_dataset):
    resum = pd.DataFrame({
        'tipus': df.dtypes.astype(str),
        'nuls': df.isna().sum(),
        'infinits': np.isinf(df.select_dtypes(include=[np.number])).sum().reindex(df.columns, fill_value=0),
        'valors_unics': df.nunique(dropna=False)
    })
    resum.index.name = f'columna ({nom_dataset})'
    return resum


def resum_numerics(df, exclou=('codi_barri',)):
    cols = [c for c in df.columns if c not in exclou]
    resum = df[cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
    resum['asimetria'] = df[cols].skew(numeric_only=True)
    return resum.sort_values('asimetria', key=lambda s: s.abs(), ascending=False)


def files_problematiques(df, dim, dataset_name):
    numeric_cols = [c for c in df.columns if c != 'codi_barri']
    problemes = []
    for col in numeric_cols:
        s = df[col]
        mask = s.isna() | np.isinf(s)
        if mask.any():
            tmp = df.loc[mask, ['codi_barri', col]].merge(
                dim[['codi_barri', 'nom_barri']],
                on='codi_barri',
                how='left'
            )
            tmp.insert(0, 'dataset', dataset_name)
            tmp.insert(1, 'variable', col)
            problemes.append(tmp.rename(columns={col: 'valor'}))
    if problemes:
        return pd.concat(problemes, ignore_index=True)
    return pd.DataFrame(columns=['dataset', 'variable', 'codi_barri', 'nom_barri', 'valor'])


def validacio_proporcions(df, dataset_name):
    resultat = []
    for col in df.columns:
        if col.startswith('pct_'):
            fora_rang = ((df[col] < 0) | (df[col] > 1)).sum()
            resultat.append({'dataset': dataset_name, 'variable': col, 'fora_rang_0_1': int(fora_rang)})
    return pd.DataFrame(resultat)


def grafics_distribucio(df, titol, nom_fitxer):
    cols = [c for c in df.columns if c != 'codi_barri']
    ncols = 3
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, cols):
        sns.histplot(df[col], kde=True, ax=ax)
        ax.set_title(col)

    for ax in axes[len(cols):]:
        ax.axis('off')

    fig.suptitle(titol, fontsize=14)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / nom_fitxer, dpi=150, bbox_inches='tight')
    plt.show()


def boxplots_validacio(df, titol, nom_fitxer):
    cols = [c for c in df.columns if c != 'codi_barri']
    ncols = 3
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, cols):
        sns.boxplot(x=df[col], ax=ax)
        ax.set_title(col)

    for ax in axes[len(cols):]:
        ax.axis('off')

    fig.suptitle(titol, fontsize=14)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / nom_fitxer, dpi=150, bbox_inches='tight')
    plt.show()

# 1. Validacio estructural

En aquest apartat comprovem que els tres datasets comparteixen l'estructura esperada per treballar amb barris de Barcelona.

In [ ]:
expected_barri_count = dim_barris['codi_barri'].nunique()
print('Nombre de barris a la dimensio:', expected_barri_count)
print('Barris unics df_2015:', df_2015['codi_barri'].nunique())
print('Barris unics df_2023:', df_2023['codi_barri'].nunique())
print('Barris unics df_deltes:', df_deltes['codi_barri'].nunique())

print()
print('Duplicats per codi_barri')
print('df_2015:', df_2015['codi_barri'].duplicated().sum())
print('df_2023:', df_2023['codi_barri'].duplicated().sum())
print('df_deltes:', df_deltes['codi_barri'].duplicated().sum())


In [ ]:
barri_sets = {
    'dimensio': set(dim_barris['codi_barri']),
    'df_2015': set(df_2015['codi_barri']),
    'df_2023': set(df_2023['codi_barri']),
    'df_deltes': set(df_deltes['codi_barri'])
}

for nom, values in barri_sets.items():
    if nom == 'dimensio':
        continue
    print()
    print(nom)
    print('Falten a la dimensio:', sorted(values - barri_sets['dimensio']))
    print('Falten al dataset:', sorted(barri_sets['dimensio'] - values))


In [ ]:
resum_estructura(df_2015, 'df_2015')

In [ ]:
resum_estructura(df_2023, 'df_2023')

In [ ]:
resum_estructura(df_deltes, 'df_deltes')

# 2. Validacio de completitud i consistencia numerica

Aquesta seccio detecta valors nuls i infinits, especialment rellevants en variables de tipus delta percentual.

In [ ]:
problemes_2015 = files_problematiques(df_2015, dim_barris, 'df_2015')
problemes_2023 = files_problematiques(df_2023, dim_barris, 'df_2023')
problemes_deltes = files_problematiques(df_deltes, dim_barris, 'df_deltes')

problemes = pd.concat([problemes_2015, problemes_2023, problemes_deltes], ignore_index=True)
problemes

In [ ]:
proporcions_2015 = validacio_proporcions(df_2015, 'df_2015')
proporcions_2023 = validacio_proporcions(df_2023, 'df_2023')

pd.concat([proporcions_2015, proporcions_2023], ignore_index=True)

In [ ]:
checks_addicionals = pd.DataFrame([
    {'dataset': 'df_2015', 'variable': 'index_gini', 'fora_rang_0_100': int(((df_2015['index_gini'] < 0) | (df_2015['index_gini'] > 100)).sum())},
    {'dataset': 'df_2023', 'variable': 'index_gini', 'fora_rang_0_100': int(((df_2023['index_gini'] < 0) | (df_2023['index_gini'] > 100)).sum())},
    {'dataset': 'df_2015', 'variable': 'preu_mitja_m2', 'negatius': int((df_2015['preu_mitja_m2'] < 0).sum())},
    {'dataset': 'df_2023', 'variable': 'preu_mitja_m2', 'negatius': int((df_2023['preu_mitja_m2'] < 0).sum())},
    {'dataset': 'df_2015', 'variable': 'import_euros', 'negatius': int((df_2015['import_euros'] < 0).sum())},
    {'dataset': 'df_2023', 'variable': 'import_euros', 'negatius': int((df_2023['import_euros'] < 0).sum())}
])
checks_addicionals

# 3. Validacio distributiva

Aquest apartat no busca interpretar el fenomen, sino detectar asimetries, escales molt diferents i outliers que puguin afectar el clustering.

In [ ]:
resum_numerics(df_2015)

In [ ]:
resum_numerics(df_2023)

In [ ]:
resum_numerics(df_deltes)

In [ ]:
grafics_distribucio(df_2015, 'Distribucions dels atributs - 2015', 'validacio_distribucions_2015.png')

In [ ]:
grafics_distribucio(df_2023, 'Distribucions dels atributs - 2023', 'validacio_distribucions_2023.png')

In [ ]:
boxplots_validacio(df_2015, 'Boxplots de validacio - 2015', 'validacio_boxplots_2015.png')

In [ ]:
boxplots_validacio(df_2023, 'Boxplots de validacio - 2023', 'validacio_boxplots_2023.png')

# 4. Validacio temporal i deltes

El dataset de deltes es especialment critic perque pot amplificar problemes derivats de valors molt baixos o iguals a zero a l'any base.

In [ ]:
resum_deltes_problematiques = pd.DataFrame({
    'nuls': df_deltes.isna().sum(),
    'infinits': np.isinf(df_deltes.select_dtypes(include=[np.number])).sum().reindex(df_deltes.columns, fill_value=0)
}).query('nuls > 0 or infinits > 0')
resum_deltes_problematiques

In [ ]:
problemes_deltes.sort_values(['variable', 'codi_barri'])

In [ ]:
variables_delta = [c for c in df_deltes.columns if c != 'codi_barri']
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df_deltes[variables_delta].replace([np.inf, -np.inf], np.nan).isna(), cbar=False, ax=ax)
ax.set_title('Mapa de valors problematics al dataset de deltes (NaN i inf transformats a NaN)')
ax.set_xlabel('Variables')
ax.set_ylabel('Files / barris')
plt.tight_layout()
plt.show()

# 5. Decisions per al clustering

Aquesta seccio recull decisions metodologiques i punts pendents abans de construir el model de clustering.

In [ ]:
decisions = pd.DataFrame([
    {
        'aspecte': 'df_2015 i df_2023',
        'estat': 'valid',
        'comentari': 'No presenten nuls ni duplicats i mantenen una estructura coherent per barri.'
    },
    {
        'aspecte': 'Variables pct_*',
        'estat': 'documentar',
        'comentari': 'Estan en escala 0-1. Cal deixar-ho explicit abans del clustering per evitar confusions d interpretacio.'
    },
    {
        'aspecte': 'df_deltes',
        'estat': 'revisar',
        'comentari': 'Hi ha NaN i inf en variables calculades com a canvis percentuals sobre bases inicials nul.les o molt petites.'
    },
    {
        'aspecte': 'Escala de variables',
        'estat': 'revisar',
        'comentari': 'Renda, gini, preu i taxes per 1000 hab treballen en escales diferents. Abans del clustering caldra escalar.'
    },
    {
        'aspecte': 'Outliers',
        'estat': 'revisar',
        'comentari': 'Algunes variables urbanes presenten cues llargues i outliers. Conve valorar robust scaling o transformacions.'
    }
])

decisions

## Conclusions

Si la clustering es fara sobre els datasets de nivell 2015 o 2023, els fitxers semblen prou consistents per continuar, tot i que caldra escalar les variables.

Si la clustering ha d incorporar els deltes, primer cal tornar a `feature_engineering.ipynb` per redefinir o tractar els canvis percentuals que generen `NaN` i `inf`.